###Download Data

Download the applicant_data from the S3 bucket.

In [ ]:
import os
import boto3
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def download_applicant_data(
    bucket_name="grad-cafe-cc",
    key="applicant_data.json",
    local_filename="applicant_data_SM.json",
    local_dir="."
):
    s3 = boto3.client("s3")
    local_path = os.path.join(local_dir, local_filename)

    try:
        s3.download_file(bucket_name, key, local_path)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"Downloaded s3://{bucket_name}/{key} -> {local_path} ({size_kb:.1f} KB)")
        return local_path
    except s3.exceptions.NoSuchKey:
        raise FileNotFoundError(f"'{key}' not found in bucket '{bucket_name}'.")
    except s3.exceptions.NoSuchBucket:
        raise FileNotFoundError(f"Bucket '{bucket_name}' does not exist or is not accessible.")
    except Exception as e:
        raise RuntimeError(f"Failed to download '{key}' from '{bucket_name}': {e}")

def save_and_upload_to_s3(df, local_filename, s3_bucket, s3_key=None, orient='records'):
    """
    Save a DataFrame to a local JSON file and upload it to S3.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to save and upload.
    local_filename : str
        Local path/filename to save the JSON file to (e.g., 'cleaned_gradcafe.json').
    s3_bucket : str
        Name of the target S3 bucket.
    s3_key : str, optional
        Destination path/filename inside the S3 bucket. Defaults to local_filename if not provided.
    orient : str, optional
        Orientation for df.to_json (default 'records').

    Returns
    -------
    bool
        True if upload succeeded, False otherwise.
    """
    if s3_key is None:
        s3_key = local_filename

    # Save locally
    df.to_json(local_filename, orient=orient, date_format='iso', indent=2)
    print(f"Saved dataframe to {local_filename}")

    # Upload to S3
    s3_client = boto3.client('s3')
    try:
        s3_client.upload_file(local_filename, s3_bucket, s3_key)
        print(f"Upload successful: s3://{s3_bucket}/{s3_key}")
        return True
    except Exception as e:
        print(f"Upload failed: {e}")
        return False

###1. Load and Clean
This code creates a pandas dataframe and loads it with the information retrieved from the S3 bucket.
It then uses pandas' head() function to print the first lines.

In [ ]:
download_applicant_data()

df = pd.read_json("applicant_data_SM.json")
df.head()

 raw_df = df.copy()

###1. Load and Clean
Pandas has useful tools built in for the size of the dataframe, so we call shape() on the dataframe to get the count for the rows and columns. Then the dtypes() function gives us the columns and their current datatypes.

The cell afterward searches for the number of fields marked with a literal NaN (even though there shouldn't be any) and for fields marked with "None", which was the default I used for no data (pandas won't recognize the string literal). It creates a list of True and False values for all of the fields I am looking for and stores these values in missing_count, then compares the sum of these missing values against the number of rows obtain just prior to get the percentage, storing the values in missing_pct. Finally, the compiled data is saved into a new dataframe, that is printed to see, then saved in the cell following it (just for cleanliness).

In [ ]:
n_rows, n_cols = df.shape

print(f"Total rows: {n_rows}")
print(f"Total columns: {n_cols}")

print("\nColumns and datatypes:")
print(df.dtypes)

In [ ]:
# Setup for change tracking
row_tally = n_rows
num_rows_removed = 0
total_rows_removed = 0

# Count of true NaN plus the literal "None" string as missing
missing_count = df.isna().sum() + (df == "None").sum()
missing_pct = (missing_count / n_rows * 100).round(2)

missingness_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": missing_count.values,
    "missing_pct": missing_pct.values
})

missingness_summary

In [ ]:
# Save the missingness summary
missingness_summary.to_csv("missingness_summary.csv", index=False)
print("Saved missingness_summary.csv")

### 2. Clean and Standardize the Dataset (Part 1, String Cleaning, Program and University Data)

The first step takes a slice from the dataframe that includes only rows where the program column contains anything other than "None" (or NaN as a carry along, but that is a desired result). This shouldn't actually do anything to the dataframe, because according to the missingness table, there are no rows with a None or NaN program field.

We skip the split step of Program and University, as this was already performed by the scraper.

In step two, we find every column that has a dtype of object or string so we know which columns to clean.

Apply() is one of the cleanest methods to step element-wise through a column, so we need a function to perform this. To accomplish this, we chain strip() to clean both sides with a replace regex that will find any length of continuous whitespace and convert it into a single space. This shouldn't touch the capitalization that was performed in the scraping phase, so meaningful capitalization will remain preserved.

Then we apply the string cleaning to all of the relevant columns.

Finally, in step 5 we create a series that includes the uniques counts, then from that, we slice out a series that only includes program that occur less than 3 times.

In [ ]:
# 1. Filter out rows where 'program' is 'None'
# (This also drops rows where the value is NaN, as NaN != 'None' evaluates to False)
# Use program instead of llm-generated-program in case the LLM turned 'None' into
# something else
df = df[df['program'] != 'None']

# Update status
print("Row check after Program adjustments")
print("==========")
num_rows_removed = row_tally - len(df) + total_rows_removed
row_tally = row_tally - num_rows_removed
total_rows_removed = total_rows_removed + num_rows_removed
print(f"Previous rows: {n_rows}, Current rows: {row_tally}")
print(f"Rows removed this pass: {num_rows_removed}")

# 2. Identify columns that contain text (string or object types)
# We only want to clean text columns, not numeric ones
text_columns = df.select_dtypes(include=['object', 'string']).columns

# 3. Define a function to clean the text
def clean_text_data(series):
    """
    Removes leading/trailing whitespace and collapses internal whitespace.
    Returns the cleaned text without changing capitalization.
    """
    return series.str.strip().str.replace(r'\s+', ' ')

# 4. Apply the cleaning function to the identified text columns
df[text_columns] = df[text_columns].apply(clean_text_data)

# 5. Count the frequency of each unique program and filter for < 3
program_counts = df['llm-generated-program'].value_counts()
low_frequency_programs = program_counts[program_counts < 3]

print(f"Number of low frequency Programs: { len(low_frequency_programs) }")
low_frequency_programs

###2. Clean and Standardize the Dataset (Part 2, Outcome and Decision Date Adjustments)

Convert date_added to datetime

Scraping already split the Status into two fields, status and status_date, so we rename status to outcome and status_date to decision_date.

Scraped data came in with "Wait listed" as two seperate words, so adjust to combine them.

The instructions didn't specify what to do with rows that contained none of the listed values, so I removed any rows without a matching outcome.

Convert "None" decision_date entries to NaN

Then, create a mask of non-NaN entries, set all matching decision_date entries to append the last four characters of the term (the year)

In [ ]:
# 1. Convert date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# 2. Rename 'status' to 'outcome' and 'status_date' to 'decision_date'
df = df.rename(columns={'status': 'outcome'})
df = df.rename(columns={'status_date': 'decision_date'})

# 3. Clean the text: Convert "Wait listed" to "Waitlisted" and strip whitespace
df['outcome'] = df['outcome'].str.strip().str.replace('Wait listed', 'Waitlisted', case=False)

# 4. Define valid outcomes and filter out outliers
valid_outcomes = ['Waitlisted', 'Interviewed', 'Rejected', 'Accepted']
df = df[df['outcome'].isin(valid_outcomes)]

# Update status
print("Row check after Status adjustments")
print("==========")
num_rows_removed = row_tally - len(df) + total_rows_removed
row_tally = row_tally - num_rows_removed
total_rows_removed = total_rows_removed + num_rows_removed
print(f"Previous rows: {n_rows}, Current rows: {row_tally}")
print(f"Rows removed this pass: {num_rows_removed}")

# 5. Replace any instances of "None" (string) or Python None with NaN
df['decision_date'] = df['decision_date'].replace('None', pd.NA)

# 6. For rows that are not NaN, append the year from the 'term' column
# Create a boolean mask for valid decision dates
valid_mask = df['decision_date'].notna()

# Append the last 4 chars of 'term' (the year) to the decision_date
df.loc[valid_mask, 'decision_date'] = (
    df.loc[valid_mask, 'decision_date'] + ' ' + df.loc[valid_mask, 'term'].str[-4:]
)

# Convert decision_date to datetime, because it should be
df['decision_date'] = pd.to_datetime(df['decision_date'], errors='coerce')

# Validate our changes
print(df[['date_added', 'outcome', 'term', 'decision_date']].head(5))

###2. Clean and Standardize the Dataset (Part 3, Degree, US/International, and float conversions)

If there are entries with no degree, it will be faster ultimately to drop them before running the map

Create a map to convert lower-cased (to standardize) versions of the degrees to expected formats. I know that "Masters" is the format used by gradcafe, so we'll include that conversion in the check.

Then we can re-use the mapping values to drop any other non-matches.

4-6 Same process to make sure spelling is standardized and only proper rows are kept

Create a list of the fields that need conversion, then iterate to convert with to_numeric() with coerce, which will turn non-numeric strings into NaN

In [ ]:
# 1. Drop 'None' degree rows
df = df[df['degree'] != 'None']

# 2. Map/Standardize
# This converts "phd", "PHD", "MASTERs" into "PhD" and "Master's"
mapping = {
    "masters": "Master's",
    "master's" : "Master's",
    "phd": "PhD",
    "psyd": "PsyD"
}
df['degree'] = df['degree'].str.lower().map(mapping)

# 3. Filter on the standardized values
# Now the filter looks for "Master's", "PhD", and "PsyD"
df = df[df['degree'].isin(mapping.values())]

# Update status
print("Row check after Degree adjustments")
print("==========")
number_rows_cleaned = row_tally - len(df)
row_tally = row_tally - number_rows_cleaned
print(f"Previous rows: {n_rows}, Current rows: {row_tally}")
print(f"Rows removed this pass: {number_rows_cleaned}")

# 4. Drop 'None' degree rows
df = df[df['US/International'] != 'None']

# 5. Map/Standardize
# This converts "phd", "PHD", "MASTERs" into "PhD" and "Master's"
mapping = {
    "international": "International",
    "american": "American",
    "other": "Other"
}
df['US/International'] = df['US/International'].str.lower().map(mapping)

# 6. Filter on the standardized values
# Now the filter looks for "Master's", "PhD", and "PsyD"
df = df[df['US/International'].isin(mapping.values())]

# 7. Define the columns that need conversion and convert them
numeric_cols = ['gpa', 'gre', 'grev', 'greaw']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Update status
print("Row check after US/International adjustments")
print("==========")
num_rows_removed = row_tally - len(df) + total_rows_removed
row_tally = row_tally - num_rows_removed
total_rows_removed = total_rows_removed + num_rows_removed
print(f"Previous rows: {n_rows}, Current rows: {row_tally}")
print(f"Rows removed this pass: {num_rows_removed}")

print(df[['gpa', 'gre', 'grev', 'greaw']].head(5))

###Mid cleaning values check

In [ ]:
n_rows_new, n_cols_new = df.shape

print(f"Previous rows: {n_rows}, Current rows: {n_rows_new}")
print(f"Previous columns: {n_cols}, Current columns: {n_cols_new}")
print(f"Total rows removed: {total_rows_removed}")

print("\nColumns and datatypes:")
print(df.dtypes)

###2. Clean and Standardize the Dataset (Part 4, New Columns)

The three new gpa/gre/grev columns are all basically the same structure, create new column by using boolean logic, comparing the values to generate a True or False, then converting that value to its integer equivalent (1 or 0) to store in the new column.

For 2, create the new column that is the difference between the post date and the acceptance date. It occured to me that someone could technically add all of the information at one time, effectively creating a negative number of days, which is not useful. In this case, we convert the data to NaN, since we can't really know what the actual time difference is.

Create a list of bins and labels for those bins, then cut the data into the bins. Unknown values are automatically converted to NaN in this process, so everything marked NaN can then safely be binned as Unknown.

Took the harder route and compared the term against the assumed application date. Rebuilt this three times to maintain vectorized operations (which probably won't make a huge difference on 50000 rows). Operations are performed using pandas or numpy to keep the array aware operations. Decided to keep the term start date (as a date) and the number of days before the term in case it was useful for statistics later.

In [ ]:
# 1. Create new 'has_valid' columns
# GPA
df['has_valid_gpa'] = ((df['gpa'].notna()) & (df['gpa'] >= 0) & (df['gpa'] <= 4.0)).astype(int)
# GRE
df['has_valid_gre'] = ((df['gre'].notna()) & (df['gre'] > 0)).astype(int)
# GRE V
df['has_valid_gre_v'] = ((df['grev'].notna()) & (df['grev'] > 0)).astype(int)

# Do some change checking and count the number of invalid rows
validity_columns = ['has_valid_gpa', 'has_valid_gre', 'has_valid_gre_v']

invalid_counts = (df[validity_columns] == 0).sum()
invalid_counts.index = ['gpa', 'gre', 'grev']  # relabel for readability

print(invalid_counts)

# 2. Create days_to_decision as date difference in days from decision_date and date_added
df['days_to_decision'] = (df['decision_date'] - df['date_added']).dt.days

# Had the question of whether or not someone created the entry for past information
# making the difference negative, so I removed them (set them to NaN) so they won't
# affect any statistics
df.loc[df['days_to_decision'] < 0, 'days_to_decision'] = np.nan

# 3. Bin the days_to_decision data using np.select
conditions = [
    df['days_to_decision'].isna(),
    df['days_to_decision'] <= 30,
    df['days_to_decision'] <= 60,
]
choices = ['Unknown', '0-30 days', '31-60 days']

df['decision_speed'] = np.select(conditions, choices, default='61+ days')

# Add "Unknown" as a category, then fill NaN values with it
df['decision_speed'] = df['decision_speed'].cat.add_categories('Unknown').fillna('Unknown')

# 4. Trying to keep this to pandas vectorized functions
# Split "term" into season and year columns in series (vectorized string ops)
term_split = df['term'].str.split(' ', expand=True)
season = term_split[0]
year = pd.to_numeric(term_split[1], errors='coerce')

# Assume start months for possible seasons, schools vary but only slightly
# Map season -> start month (vectorized map, no loop)
season_start_month = {'Fall': 9, 'Spring': 1, 'Summer': 6, 'Winter': 1}
month = season.map(season_start_month)

# Assume first day of each month is the start day
# Build term_start_date vectorized using pd.to_datetime on a dict of arrays
df['term_start_date'] = pd.to_datetime(
    {'year': year, 'month': month, 'day': 1},
    errors='coerce')

# Days before term (vectorized subtraction)
df['days_before_term'] = (df['term_start_date'] - df['date_added']).dt.days

# Bin using np.select instead of .apply (vectorized conditional assignment)
# Default to 'Late Cycle' because it should comprise the greatest number of months
conditions = [
    df['days_before_term'].isna(),
    df['days_before_term'] >= 150,
    df['days_before_term'] >= 60,
]
choices = ['Unknown', 'Early Cycle', 'Mid Cycle']

df['application_season'] = np.select(conditions, choices, default='Late Cycle')

###5. Summary Statistics and Outlier Analysis (Part 1, Required Statistics)

Use vectorized options and describe() to get values. 50th percentile and median are calculated internally to pandas in the same way, so copy 50th percentile to median. Reorder everything just to match. Save to csv, then print it just to verify it.

In [ ]:
columns_to_summarize = ['gpa', 'gre', 'grev', 'greaw']

summary_df = df[columns_to_summarize].describe(percentiles=[0.25, 0.5, 0.75]).T
summary_df = summary_df.rename(columns={'25%': '25th_percentile', '50%': '50th_percentile', '75%': '75th_percentile'})

# Add median as its own column (identical to 50th_percentile)
summary_df['median'] = summary_df['50th_percentile']

# Reorder columns to match requirements
summary_df = summary_df[['count', 'mean', 'median', 'std', 'min', 'max', '25th_percentile', '50th_percentile', '75th_percentile']]

summary_df.to_csv('summary_statistics.csv')
summary_df

###5. Summary Statistics and Outlier Analysis (Part 2, Outlier Analysis)

Per directions, uses scipy to calculate the z score, then uses the z score to create new columns. Prints the sum of the outliers to see how many are in the data. There were 137 GPA outliers and 33 GRE outliers. Which is not particularly surprising, since I knew there were out of bounds numbers in some of the rows already from previous assignments. I was going to remove all of them, but we created the 'has_valid_gpa' column that I could use to filter out the invalid GPA entries. Unfortunately, since we didn't do an upper bound check on the GRE, I can't use the 'has_valid_gre' for the GRE. Even trying to place that upper bound is problematic, since a data entry field of 'GRE' could have been interpreted to mean the combination of the GRE-V and GRE-Q, which is apparently a common way to list one's score. A score between 260 and 340 would tell us this is what happened, but wouldn't tell us what the distribution of points would be, so I can't parse it out. So, I think GPA stays and GRE scores will have to be switched to NaN to at least tame any big swings.

In [ ]:
# Compute z-scores, ignoring NaN values so they don't break the calculation
gpa_zscore = stats.zscore(df['gpa'], nan_policy='omit')
gre_zscore = stats.zscore(df['gre'], nan_policy='omit')

# Flag outliers where |z-score| > 3 (NaN comparisons naturally return False)
df['gpa_outlier'] = pd.Series(np.abs(gpa_zscore) > 3, index=df.index).fillna(False)
df['gre_outlier'] = pd.Series(np.abs(gre_zscore) > 3, index=df.index).fillna(False)

# Print counts
print(f"GPA outliers: {df['gpa_outlier'].sum()}")
print(f"GRE outliers: {df['gre_outlier'].sum()}")

In [ ]:
df.loc[df['gre_outlier'], 'gre'] = np.nan
df.loc[df['gre_outlier'], 'gre_outlier'] = False

# Need to recalculate has_valid_gre now
df['has_valid_gre'] = ((df['gre'].notna()) & (df['gre'] > 0)).astype(int)

df['gre_outlier'].sum()  # should now be 0

###6. Statistical Analysis with SciPy (Part A, Correlation Analysis)

Masked out the values that are invalid and ran the correlation on them using scipy.



In [ ]:
# Conduct GRE-GREV correlation
valid_mask = (df['has_valid_gre'] == 1) & (df['has_valid_gre_v'] == 1)

correlation = df.loc[valid_mask, 'gre'].corr(df.loc[valid_mask, 'grev'])

r, p_value = stats.pearsonr(df.loc[valid_mask, 'gre'], df.loc[valid_mask, 'grev'])
print(f"Pearson r: {r:.4f}, p-value: {p_value:.4g}")

# Conduct GPA-GRE correlation
valid_mask = (df['has_valid_gpa'] == 1) & (df['has_valid_gre'] == 1)

correlation = df.loc[valid_mask, 'gpa'].corr(df.loc[valid_mask, 'gre'])

r, p_value = stats.pearsonr(df.loc[valid_mask, 'gpa'], df.loc[valid_mask, 'gre'])
print(f"Pearson r: {r:.4f}, p-value: {p_value:.4g}")

###6. Statistical Analysis with SciPy (Part B, Group Comparison)

Pull out a set of all valid GPAs, then pull out sets for the accepted and rejected GPAs from the valid entries. Using SciPy, we can then run the mannwhitneyu() comparison on the groups, returning the u_stat, and p_value.

In [ ]:
# Filter to valid GPA rows only, split into two groups
valid_gpa = df[df['has_valid_gpa'] == 1]

accepted_gpa = valid_gpa.loc[valid_gpa['outcome'] == 'Accepted', 'gpa']
rejected_gpa = valid_gpa.loc[valid_gpa['outcome'] == 'Rejected', 'gpa']

# Mann-Whitney U test (non-parametric, doesn't assume normal distribution)
u_stat, p_value = stats.mannwhitneyu(accepted_gpa, rejected_gpa, alternative='two-sided')

print(f"Test used: Mann-Whitney U")
print(f"U statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.4g}")
print(f"Accepted group: n={len(accepted_gpa)}, median GPA={accepted_gpa.median():.3f}")
print(f"Rejected group: n={len(rejected_gpa)}, median GPA={rejected_gpa.median():.3f}")

###6. Statistical Analysis with SciPy (Part C, Categorical Association)


In [ ]:
# Build contingency table
contingency_table = pd.crosstab(df['degree'], df['outcome'])
print("Contingency Table:")
print(contingency_table)

# Chi-square test of independence
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4g}")
print(f"Degrees of freedom: {dof}")

###7. Produce Visualizations

Overview for all plots: Most of creating the plots is code to just set everything up and doesn't require any specifically complicated code. I broke each plot into a seperate cell, just to make it clearer which plot I was working on as I worked through these.

Plots 1 & 2 are effectively the same process and uses polyfit and polyval to generate the best fit lines

Plot 3 created a crosstab to and generated the plots.

Plot 4 Sets the bins to every 3 days (3D) and sets the plot up (had to replot this a couple times to adjust the table properly)

Plot 5 has nothing particularly special, just made sure to use only "valid" data entries.

Plot 6 I had to look this one up, because I've never done a heat map. I pulled some examples from Datacamp and Dive Into Data Science by Bradford Tuckfield from No Starch Press (which is a great general python data science book).

In [ ]:
# ------------------------------------------------------------------
# 1. Scatter: non-zero GRE vs non-zero GRE V (with best-fit line)
# ------------------------------------------------------------------
mask = (df['gre'] > 0) & (df['grev'] > 0)
x = df.loc[mask, 'gre']
y = df.loc[mask, 'grev']

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, alpha=0.3, s=10, label='Applicants')

# Best-fit line
coeffs = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = np.polyval(coeffs, x_line)
ax.plot(x_line, y_line, color='red', linewidth=2,
        label=f'Best fit: y = {coeffs[0]:.3f}x + {coeffs[1]:.2f}')

ax.set_title('GRE Quantitative vs GRE Verbal Scores')
ax.set_xlabel('GRE Quantitative Score (130-170)')
ax.set_ylabel('GRE Verbal Score (130-170)')
ax.legend()
fig.tight_layout()
fig.savefig('GRE-vs-GRE-V.png', dpi=150)
print('GRE-vs-GRE-V.png saved!')
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# 2. Scatter: non-zero GPA (4.0 scale) vs non-zero GRE (with best-fit line)
# ------------------------------------------------------------------
mask = (df['gpa'] > 0) & (df['gpa'] <= 4.0) & (df['gre'] > 0)
x = df.loc[mask, 'gpa']
y = df.loc[mask, 'gre']

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, alpha=0.3, s=10, label='Applicants')

coeffs = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = np.polyval(coeffs, x_line)
ax.plot(x_line, y_line, color='red', linewidth=2,
        label=f'Best fit: y = {coeffs[0]:.3f}x + {coeffs[1]:.2f}')

ax.set_title('GPA vs GRE Quantitative Score')
ax.set_xlabel('GPA (4.0 scale)')
ax.set_ylabel('GRE Quantitative Score (130-170)')
ax.legend()
fig.tight_layout()
fig.savefig('GPA-vs-GRE.png', dpi=150)
print('GPA-vs-GRE.png saved!')
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# 3. Bar chart: Degree count broken down by US/International
# ------------------------------------------------------------------
degree_counts = pd.crosstab(df['degree'], df['US/International'])

fig, ax = plt.subplots(figsize=(9, 6))
degree_counts.plot(kind='bar', ax=ax)

ax.set_title('Applicant Count by Degree Type and Residency Status')
ax.set_xlabel('Degree Type')
ax.set_ylabel('Number of Applicants')
ax.legend(title='US/International')
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig('Degree-vs-International.png', dpi=150)
print('Degree-vs-International.png saved!')
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# 4. Histogram: Accepted applicants by decision_date, binned every 3 days
# ------------------------------------------------------------------
accepted = df[(df['outcome'] == 'Accepted') & (df['decision_date'].notna())]

date_min = accepted['decision_date'].min()
date_max = accepted['decision_date'].max()
bins = pd.date_range(start=date_min, end=date_max + pd.Timedelta(days=3), freq='3D')

# Convert dates to matplotlib's internal numeric format explicitly
bins_numeric = mdates.date2num(bins)
data_numeric = mdates.date2num(accepted['decision_date'])

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(data_numeric, bins=bins_numeric, edgecolor='black', linewidth=0.5,
        label='Accepted Applicants')

# Explicitly set x-axis limits to match the actual data range (no extra padding)
ax.set_xlim(bins_numeric[0], bins_numeric[-1])
ax.margins(x=0)

# Format x-axis as dates
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

ax.set_title('Acceptances Over Time (3-Day Intervals)')
ax.set_xlabel('Decision Date')
ax.set_ylabel('Number of Acceptances')
fig.autofmt_xdate(rotation=45)
ax.legend()
fig.tight_layout()
fig.savefig('Acceptances-over-Time.png', dpi=150)
print('Acceptances-over-Time.png saved!')
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# 5. Boxplot: GPA distribution by outcome
# ------------------------------------------------------------------
valid_gpa = df[df['has_valid_gpa'] == 1]
outcome_order = ['Accepted', 'Rejected', 'Waitlisted']
data_by_outcome = [valid_gpa.loc[valid_gpa['outcome'] == o, 'gpa'] for o in outcome_order]

fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(data_by_outcome, labels=outcome_order)

ax.set_title('GPA Distribution by Admission Outcome')
ax.set_xlabel('Outcome')
ax.set_ylabel('GPA (4.0 scale)')
fig.tight_layout()
fig.savefig('GPA-by-Outcome.png', dpi=150)
print('GPA-by-Outcome.png saved!')
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# 6. Correlation heatmap: GPA, GRE, GRE V, GRE AW
# ------------------------------------------------------------------
numeric_cols = ['gpa', 'gre', 'grev', 'greaw']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)

# Tick labels
labels = ['GPA', 'GRE', 'GRE V', 'GRE AW']
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

# Annotate each cell with its correlation value
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', color='black')

ax.set_title('Correlation Heatmap: GPA, GRE, GRE V, GRE AW')
fig.colorbar(im, ax=ax, label='Pearson Correlation Coefficient')
fig.tight_layout()
fig.savefig('Numeric-Correlation-Heatmap.png', dpi=150)
print('Numeric-Correlation-Heatmap.png saved!')
plt.show()
plt.close(fig)

###Additional Analytics

There were some questions that needed some calculation so I put them here.

Based on the charts I wanted to break down question 1 a little further and see if there was any large difference in the scores that was skewing the data.

The GPA is already bounds checked with 'has_valid_gpa' so should be an accurate reflection.

In [ ]:
# 1. What is the average GRE score of an applicant who has been accepted into a PhD program?
base_mask = (df['has_valid_gre'] == 1) & (df['outcome'] == 'Accepted') & (df['degree'] == 'PhD')

# Test 1: Original question, no range restriction beyond has_valid_gre
gre_all = df.loc[base_mask, 'gre']

# Test 2: Subset in combined GRE-Q + GRE-V range (260-340)
combined_mask = base_mask & (df['gre'] >= 260) & (df['gre'] <= 340)
gre_combined = df.loc[combined_mask, 'gre']

# Test 3: Subset in single-section GRE-Q range only (130-170)
single_section_mask = base_mask & (df['gre'] >= 130) & (df['gre'] <= 170)
gre_single = df.loc[single_section_mask, 'gre']

# Report all three
print("=== Test 1: Original (all valid GRE, no range filter) ===")
print(f"Average GRE: {gre_all.mean():.2f}, n={len(gre_all)}")

print("\n=== Test 2: Combined GRE-Q + GRE-V range (260-340) ===")
print(f"Average GRE: {gre_combined.mean():.2f}, n={len(gre_combined)}")

print("\n=== Test 3: Single-section GRE-Q range (130-170) ===")
print(f"Average GRE: {gre_single.mean():.2f}, n={len(gre_single)}")

# 2. What is the average GPA of an applicant who has been accepted into a Master’s program?
mask = (df['has_valid_gpa'] == 1) & (df['outcome'] == 'Accepted') & (df['degree'] == "Master's")

avg_gpa_masters_accepted = df.loc[mask, 'gpa'].mean()

print(f"Average GPA for accepted Master's applicants: {avg_gpa_masters_accepted:.3f}")
print(f"Based on n={mask.sum()} applicants")

###9. Save the Cleaned Dataset and Return It to S3


In [ ]:
save_and_upload_to_s3(
    df=df,
    local_filename='cleaned_gradcafe.json',
    s3_bucket='grad-cafe-cc'
)